# Galilean IMU preintegration with a left-invariant error

This guide develops the left-invariant Galilean IMU preintegration used by GTSAM: a direct-product symmetry, compatible left actions, exact zero-order-held Galilean increments, adjoint covariance transport, right-applied bias correction, and exact constant-rate rotating-frame prediction.

The construction is motivated by Delama, Fornasier, Mahony, and Weiss, [*Equivariant IMU Preintegration with Biases: a Galilean Group Approach*](https://arxiv.org/abs/2411.05548), but deliberately uses a different symmetry and perturbation convention. It closes on the physical six-dimensional IMU input and bias in accelerometer-then-gyroscope order, and it induces the same left-invariant, component-wise right-retracted endpoint error used by GTSAM.

For background, see the [`EquivariantFilter` guide](EKF-variants.md#equivariantfilter), the [`Gal3` guide](../../geometry/doc/Gal3.ipynb), the [`Gal3ImuEKF` guide](Gal3ImuEKF.ipynb), the [`NavState` guide](NavState.ipynb), and the standard [`ImuFactor` guide](ImuFactor.ipynb). The companion [`NEES comparison`](GalileanImuFactorNEES.ipynb) evaluates all four GTSAM preintegration backends under identical high-dynamic IMU samples.

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/navigation/doc/GalileanImuFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import numpy as np
import gtsam

## 1. Conventions: left-invariant error means an update on the right

We fix the convention before introducing the dynamics. For a matrix Lie group $G$ with Lie algebra $\mathfrak g$, GTSAM uses

| Operation | Definition | Consequence |
|---|---|---|
| Retraction | $X\oplus\delta = X\operatorname{Exp}(\delta)$ | A tangent increment is applied on the **right**. |
| Local coordinates | $\operatorname{Local}(X,Y)=\operatorname{Log}(X^{-1}Y)$ | The displacement is expressed in the local frame of $X$. |
| Error about $\hat X$ | $X=\hat X\operatorname{Exp}(\epsilon)$ | $\epsilon=\operatorname{Local}(\hat X,X)$. |

The group error $\hat X^{-1}X$ is **left-invariant**: replacing both arguments by $AX$ and $A\hat X$ leaves it unchanged. Thus, throughout this guide, *left-invariant error* and *right perturbation* describe the same convention.

Delama et al.'s motivating construction instead uses the navigation error $\Upsilon\hat\Upsilon^{-1}$, which is right-invariant, together with a left-applied first-order correction. Reversing only the error while retaining their right action would not produce an invariant EqF error; the direct-product left action introduced in Section 5 is what induces $\hat\Upsilon^{-1}\Upsilon$.

> **Do not conflate multiplication sides.** `ActionType::Left` below describes how the symmetry moves the physical state and input. GTSAM's `EquivariantFilter` prediction still composes the lifted increment on the right, while its measurement update composes the innovation on the left. The side of a symmetry action, the side of a group increment, and the invariance of an error are related choices, but they are not synonyms.

## 2. Physical state

The preintegration state consists of the accumulated Galilean motion and the physical six-axis IMU bias:

$$
\mathcal M=\mathrm{Gal}(3)\times\mathbb R^6,\qquad
\xi_k=(\Upsilon_k,\beta_k),\qquad
\beta_k=\begin{bmatrix}b_{a,k}\\b_{\omega,k}\end{bmatrix}.
$$

Here $\Upsilon_k=(\Delta R_k,\Delta v_k,\Delta p_k,\Delta t_k)$ is the motion being preintegrated. The second component is exactly `gtsam::imuBias::ConstantBias`; its vector representation stores accelerometer bias before gyroscope bias. There are no additional bias coordinates. Bias perturbations and bias-Jacobian columns retain this ordering throughout.

## 3. The Galilean motion component

A Galilean element stores rotation, velocity, position, and elapsed time:

$$
X=(R,v,p,t)
\quad\longleftrightarrow\quad
\mathbf X=
\begin{bmatrix}
R & v & p\\
0 & 1 & t\\
0 & 0 & 1
\end{bmatrix}.
$$

Matrix multiplication gives

$$
(R_1,v_1,p_1,t_1)(R_2,v_2,p_2,t_2)
=\left(R_1R_2,\;v_1+R_1v_2,\;p_1+R_1p_2+t_2v_1,\;t_1+t_2\right).
$$

The term $t_2v_1$ is the defining Galilean coupling: during the second interval, the velocity accumulated by the first interval advances position. The inverse is

$$
(R,v,p,t)^{-1}=\left(R^\top,-R^\top v,-R^\top(p-tv),-t\right).
$$

Using GTSAM's tangent ordering, an algebra element is

$$
x=(\omega,\nu,\rho,\alpha)\in\mathbb R^{10},
\qquad
x^\wedge=
\begin{bmatrix}
\omega^\wedge & \nu & \rho\\
0 & 0 & \alpha\\
0 & 0 & 0
\end{bmatrix}.
$$

For an integrated IMU increment, $\omega$ is an angle, $\nu$ a velocity increment, $\rho$ a position increment, and $\alpha$ an elapsed time. Before multiplication by $\Delta t$, the corresponding rate vector has units $({\rm rad/s}, {\rm m/s^2}, {\rm m/s}, 1)$.

### Exponential, Jacobians, and adjoint

Let $J_L(\omega)$ be the $SO(3)$ left Jacobian and $\Gamma_2(\omega)$ its second integral. Then

$$
\operatorname{Exp}(x)=
\left(
\operatorname{Exp}(\omega),
J_L(\omega)\nu,
J_L(\omega)\rho+\alpha\Gamma_2(\omega)\nu,
\alpha
\right).
$$

The $\alpha\Gamma_2(\omega)\nu$ term produces the familiar $\tfrac12a\Delta t^2$ when rotation is zero. For the complete 10D algebra, define the right Jacobian $J_R(x)$ by

$$
\operatorname{Exp}(x+\delta x)
=\operatorname{Exp}(x)
 \operatorname{Exp}(J_R(x)\delta x)+O(\|\delta x\|^2).
$$

This is the Jacobian required when both the state perturbation and the increment are applied on the right. It is related to the complete Galilean left Jacobian by

$$J_R(x)=\operatorname{Ad}_{\operatorname{Exp}(-x)}J_L(x).$$

For $X=(R,v,p,t)$, the adjoint in the ordering $(\omega,\nu,\rho,\alpha)$ is

$$
\operatorname{Ad}_X=
\begin{bmatrix}
R & 0 & 0 & 0\\
v^\wedge R & R & 0 & 0\\
(p-tv)^\wedge R & -tR & R & v\\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

We write $\operatorname{ad}_x y=[x,y]$ for the algebra adjoint and use the identities $\operatorname{Ad}_{X^{-1}}=\operatorname{Ad}_X^{-1}$ and $\operatorname{ad}_x y=-\operatorname{ad}_y x$.

## 4. Mean preintegration

The measured input is the physical six-axis IMU sample supplied to GTSAM's `integrateMeasurement(measuredAcc, measuredOmega, deltaT)` interface:

$$
u_k=\begin{bmatrix}a_k\\\omega_k\end{bmatrix}
\in\mathcal L=\mathbb R^6.
$$

It follows the same accelerometer-then-gyroscope ordering as the bias. Holding this sample constant over an interval of duration $h$, its bias-corrected Galilean body rate is

$$
q(u_k,\beta_k)=\begin{bmatrix}
\omega_k-b_{\omega,k}\\
a_k-b_{a,k}\\
0_3\\
1
\end{bmatrix}.
$$

The exact discrete dynamics for this held input are

$$
F_h((\Upsilon_k,\beta_k),u_k)=
\left(\Upsilon_k\operatorname{Exp}(q(u_k,\beta_k)h),\;\beta_k\right).
$$

The bias is constant inside preintegration because the three-way IMU factor uses a separate factor for bias evolution. No bias-process input is part of $\mathcal L$. For a measured sample $\tilde u_k=(\tilde a_k,\tilde\omega_k)$ and linearization bias $\hat\beta_k$, let

$$
q_k=q(\tilde u_k,\hat\beta_k),\qquad
x_k=q_kh,\qquad
V_k=\operatorname{Exp}(x_k).
$$

Starting with $\hat\Upsilon_{ii}=I$, the preintegrated mean is updated on the right:

$$
\boxed{\hat\Upsilon_{i,k+1}=\hat\Upsilon_{ik}V_k}.
$$

The group law automatically performs all four component updates. In particular, old velocity advances position by $h\,\Delta v_{ik}$, while the exponential contributes the rotation-aware acceleration integral. No separate Euler position update is needed.

Gal(3) is internal to preintegration; the optimized keyframe states remain $X_i,X_j\in\mathrm{SE}_2(3)$. `NavState` stores and retracts it as $(R,p,v)$, and we define

$$
W_{ij}=\operatorname{NavState}\!\left(I,\tfrac12g\Delta t_{ij}^2,g\Delta t_{ij}\right),\qquad
\phi_{\Delta t}(R,p,v)=(R,p+v\Delta t,v),
$$

and project the corrected $\Upsilon_{ij}$ to the body-frame increment

$$
U_{ij}(\beta_i)=\operatorname{NavState}(\Delta R_{ij},\Delta p_{ij},\Delta v_{ij}).
$$

The inertial endpoint prediction is exactly

$$
\boxed{X_j=W_{ij}\,\phi_{\Delta t_{ij}}(X_i)\,U_{ij}(\beta_i)}.
$$

This $W\phi(X)U$ separation is important: gravity acts in the navigation frame on the left, autonomous coasting advances the existing position, and bias-corrected IMU motion acts in the body/local frame on the right.

## 5. The equivariant-filter construction

GTSAM's `EquivariantFilter<M, Symmetry>` separates the physical state manifold $M$ from a symmetry group $G$ that moves points of $M$. A complete model also supplies an action on the input space and a lift that reconstructs the dynamics on $G$. For this factor the template roles are

| EqF role | Galilean preintegration object |
|---|---|
| Physical state | $\xi=(\Upsilon,\beta)\in\mathcal M$ |
| Symmetry group | $Y=(\Gamma,c)\in\mathcal G_L$ |
| State action | $\phi_L:\mathcal G_L\times\mathcal M\to\mathcal M$ |
| Input action | $\psi_L:\mathcal G_L\times\mathcal L\to\mathcal L$ |
| Lift | $\lambda_L:\mathcal M\times\mathcal L\to\operatorname{Lie}(\mathcal G_L)$ |

### Direct-product symmetry group

Choose the direct product

$$
\mathcal G_L=\mathrm{Gal}(3)\times(\mathbb R^6,+),
$$

where the second factor is the additive physical bias space in GTSAM order. For $Y_1=(\Gamma_1,c_1)$ and $Y_2=(\Gamma_2,c_2)$,

$$
Y_1Y_2=(\Gamma_1\Gamma_2,c_1+c_2),\qquad
Y^{-1}=(\Gamma^{-1},-c).
$$

This group has the same dimension as $\mathcal M$, but its role is different: $\mathcal M$ contains physical states, whereas $\mathcal G_L$ supplies transformations of those states.

### Left action on the state

Define

$$
\boxed{\phi_L((\Gamma,c),(\Upsilon,\beta))
=(\Gamma\Upsilon,\beta+c)}.
$$

It is a left action because

$$
\phi_L(Y_1,\phi_L(Y_2,\xi))
=\phi_L(Y_1Y_2,\xi).
$$

In GTSAM terminology the corresponding `Symmetry` derives from `GroupAction` and declares `ActionType::Left`; its call signature is `operator()(group, state)`. Fix the reference state

$$\xi^\circ=(I,0).$$

The orbit map used by `EquivariantFilter` is especially simple:

$$
\phi_{\xi^\circ}(Y)=\phi_L(Y,\xi^\circ)=(\Gamma,c).
$$

Consequently the lifted group estimate and the physical estimate have identical coordinates,

$$\hat Y=(\hat\Upsilon,\hat\beta),\qquad
\hat\xi=\phi_{\xi^\circ}(\hat Y).$$

### Left action on the input

The compatible input action is

$$
\boxed{\psi_L((\Gamma,c),u)=u+c}.
$$

It is also a left action. The Galilean component $\Gamma$ acts trivially on the physical IMU input, while $c=(c_a,c_\omega)$ translates it in the same GTSAM order as the bias. Consequently

$$
q(u+c,\beta+c)=q(u,\beta),
$$

and direct substitution verifies system equivariance:

$$
F_h(\phi_L(Y,\xi),\psi_L(Y,u))
=\phi_L(Y,F_h(\xi,u)).
$$

The `EquivariantFilter` maps an input to the reference orbit by applying its `InputOrbit` to the inverse group estimate. Here

$$
u^\circ=\psi_L(\hat Y^{-1},\tilde u)
=\tilde u-\hat\beta.
$$

Thus the input at the EqF origin is exactly the bias-corrected body/local input; no adjoint transport is required.

### Lift and reconstruction

Use the right-trivialized lift expected by GTSAM's prediction path,

$$
\boxed{\lambda_L((\Upsilon,\beta),u)
=\left(q(u,\beta),0_6\right)
\in\mathfrak{gal}(3)\times\mathbb R^6}.
$$

The direct-product exponential gives

$$
\operatorname{Exp}_{\mathcal G_L}(\lambda_L(\xi,u)h)
=\left(\operatorname{Exp}(q(u,\beta)h),0_6\right).
$$

The lift is also where the two coordinate orderings meet. For the linearization below, record its constant derivative with respect to a physical GTSAM-ordered six-vector:

$$
E=\frac{\partial q}{\partial(u-\beta)}
=\begin{bmatrix}
0&I_3\\
I_3&0\\
0&0\\
0&0
\end{bmatrix}
\in\mathbb R^{10\times6}.
$$

This single matrix places angular rate and acceleration into Gal3's internal $(\omega,\nu,\rho,\alpha)$ order. It is a derivative of the lift, not a second representation of the bias; the fixed zero position-rate and unit clock-rate coordinates have zero derivative.

The lifted state is propagated exactly as in `EquivariantFilter::predictWithJacobian`, by composing the increment on the right and then returning through the reference orbit:

$$
Y^+=Y\operatorname{Exp}_{\mathcal G_L}(\lambda_L(\xi,u)h),\qquad
\xi^+=\phi_{\xi^\circ}(Y^+).
$$

For a left state action, this reconstruction equation is not $\phi_L(\operatorname{Exp}(\lambda h),\xi)$, which would put the Galilean increment on the wrong side. The lift is invariant under the simultaneous state and input actions:

$$
\lambda_L(\phi_L(Y,\xi),\psi_L(Y,u))
=\lambda_L(\xi,u).
$$

> **Relation to Delama et al.** Their construction closes its model over the full Galilean algebra: its input and bias coordinates lie in $\mathfrak{gal}(3)$, including virtual coordinates, and its symmetry is the semidirect product $\mathcal G_R=\mathrm{Gal}(3)\ltimes\mathfrak{gal}(3)$. It uses the right actions
>
> $$
> \phi_R((\Upsilon,b),(C,\gamma))
> =(\Upsilon C,\operatorname{Ad}_{C^{-1}}(b-\gamma)),\qquad
> \psi_R((w,\tau),(C,\gamma))
> =(\operatorname{Ad}_{C^{-1}}(w-\gamma),\operatorname{Ad}_{C^{-1}}\tau).
> $$
>
> That coherent pairing induces the invariant error $X\hat X^{-1}$. The adjoint in their input action does not preserve the six-dimensional physical IMU subspace by itself, which motivates completing the input and bias to the full algebra. The direct-product construction used here needs no adjoint transport: it is closed on the physical $\mathbb R^6$ input and bias spaces, while the lift supplies the fixed Gal3 coordinates. The group and both actions change together; this is not merely a reversal of the error.

## 6. Equivariant error and left-invariant discrete dynamics

For a left action, the EqF error is obtained by acting on the true state with the inverse group estimate:

$$
\boxed{\mathsf e_k=\phi_L(\hat Y_k^{-1},\xi_k)}
=\left(\hat\Upsilon_k^{-1}\Upsilon_k,\;\beta_k-\hat\beta_k\right).
$$

This error is invariant under the common left action: replacing $\xi$ by $\phi_L(Y,\xi)$ and $\hat Y$ by $Y\hat Y$ leaves $\mathsf e$ unchanged. Because the reference orbit is the identity, its normal coordinates are simply

$$
\epsilon_k=(e_k,\delta\beta_k)\in\mathbb R^{16},\qquad
e_k=\operatorname{Log}(\hat\Upsilon_k^{-1}\Upsilon_k),\qquad
\delta\beta_k=\beta_k-\hat\beta_k.
$$

These are exactly the error coordinates in which `EquivariantFilter::errorCovariance()` is interpreted: a tangent vector at the fixed reference $\xi^\circ$. The navigation component is also exactly GTSAM's right-retracted local coordinate.
### Exact discrete linearization

Adopt the six-dimensional measurement convention

$$
\tilde u_k=u_k+\eta_k,\qquad
\eta_k=(\eta_{a,k},\eta_{\omega,k})\in\mathbb R^6.
$$

The true bias is constant during this preintegration step. For one nominal increment define

$$
x=q(\tilde u,\hat\beta)h,\qquad
V=\operatorname{Exp}(x),\qquad
C=J_R(x)hE\in\mathbb R^{10\times6}.
$$

The true corrected physical input is $\tilde u-\hat\beta-\delta\beta-\eta$. By the definition of the complete Galilean right Jacobian, its increment satisfies

$$
V_{\mathrm{true}}
=\operatorname{Exp}\left(q(\tilde u-\eta,\hat\beta+\delta\beta)h\right)
\simeq V\operatorname{Exp}(-C(\delta\beta+\eta)).
$$

Write $\mathcal E=\hat\Upsilon^{-1}\Upsilon=\operatorname{Exp}(e)$. After one mean update,

$$
\mathcal E^+=V^{-1}\mathcal E V_{\mathrm{true}}.
$$

Transporting the old right perturbation through $V$ and retaining first-order terms gives

$$
e^+=\operatorname{Ad}_{V^{-1}}e-C\delta\beta-C\eta,\qquad
\delta\beta^+=\delta\beta.
$$

Therefore, with $\epsilon=(e,\delta\beta)\in\mathbb R^{16}$,

$$
\boxed{\epsilon^+=A_{\mathrm{LI}}\epsilon+B_{\mathrm{LI}}\eta},
$$

where

$$
\boxed{
A_{\mathrm{LI}}=
\begin{bmatrix}
\operatorname{Ad}_{V^{-1}}&-C\\
0_{6\times10}&I_6
\end{bmatrix}
\in\mathbb R^{16\times16},\qquad
B_{\mathrm{LI}}=
\begin{bmatrix}
-C\\
0_{6\times6}
\end{bmatrix}
\in\mathbb R^{16\times6}.}
$$

The upper-left block is not generally the identity: $\operatorname{Ad}_{V^{-1}}$ transports a body/local error through the next increment. The upper-right block maps the physical GTSAM-ordered bias error directly into Gal3 coordinates through $C$; no virtual bias columns are introduced.

> **Using the generic filter template literally.** For an input held constant during the sample, the continuous-time error matrix is
>
> $$
> A_c=\begin{bmatrix}-\operatorname{ad}_{q^\circ}&-E\\0&0\end{bmatrix},
> \qquad \operatorname{Exp}(A_c h)=A_{\mathrm{LI}}.
> $$
>
> The current `EquivariantFilter` automatic path rejects left actions because $D\phi_0D\lambda$ alone omits the $-\operatorname{ad}_{q^\circ}$ transport associated with a left action and a right-composed prediction. A literal filter implementation must call `predictWithJacobian` with the continuous-time $A_c$; it must not pass the already-discrete $A_{\mathrm{LI}}$, which that method would discretize again. The `GalileanImuFactor` itself does not instantiate `EquivariantFilter`; the template terminology specifies the geometry used by its preintegration.

### Covariance propagation

Let $\Sigma_k\in\mathbb R^{16\times16}$ be the covariance of $(e,\delta\beta)$. In GTSAM order, the continuous-time IMU noise density is

$$
Q_u=\operatorname{diag}(Q_a,Q_\omega)\in\mathbb R^{6\times6},
\qquad Q_d=\frac{1}{h}Q_u,
$$

where $Q_d$ is the covariance of the sampled rate noise. The first-order augmented propagation is

$$
\boxed{
\Sigma_{k+1}=A_{\mathrm{LI}}\Sigma_kA_{\mathrm{LI}}^\top
+B_{\mathrm{LI}}Q_dB_{\mathrm{LI}}^\top}.
$$

Only physical accelerometer and gyroscope noise enters this model. If a caller already supplies per-sample rather than continuous-time covariances, the $1/h$ conversion must not be applied a second time.

The three-way IMU factor does not itself constrain temporal bias evolution. It uses the navigation uncertainty conditioned on the bias linearization value, while a separate bias between-factor supplies the chosen random-walk model. Therefore initialize $\delta\beta=0$ and propagate the conditional $10\times10$ navigation covariance directly:

$$
\boxed{
\Sigma_{e,k+1}=\operatorname{Ad}_{V_k^{-1}}\Sigma_{e,k}\operatorname{Ad}_{V_k^{-1}}^\top
+C_k\frac{Q_u}{h}C_k^\top}.
$$

The deterministic $10\times6$ bias sensitivity derived next accounts for changing the optimizer's bias away from its linearization value; it is not part of this conditional covariance. A combined-bias factor could extend the input with a physical six-dimensional bias random walk and retain the full augmented covariance, without introducing virtual coordinates.

The augmented covariance above belongs to the direct-product, left-action EqF error used in GTSAM. It must not be identified directly with Delama et al.'s semidirect-product covariance, whose navigation and bias components use different frames and coordinates. For the conditioned navigation covariance consumed by the ordinary factor, the descriptions agree after the corresponding frame conversion.

## 7. Bias correction must also act on the right

Preintegration is performed once at a linearization bias $\hat\beta=(\hat b_a,\hat b_\omega)$ in GTSAM order. When optimization proposes $\hat\beta+\delta\beta$, recomputing every IMU sample would be wasteful. Define $J_k\in\mathbb R^{10\times6}$ directly by

$$
\Upsilon_k(\hat\beta+\delta\beta)
\simeq\hat\Upsilon_k(\hat\beta)
\operatorname{Exp}(J_k\delta\beta).
$$

This correction is on the right, matching the retraction convention. For $V_k=\operatorname{Exp}(x_k)$ and $C_k=J_R(x_k)hE\in\mathbb R^{10\times6}$, transport of the old correction through the new increment gives

$$
\boxed{J_{k+1}=\operatorname{Ad}_{V_k^{-1}}J_k-C_k},
\qquad J_i=0.
$$

The minus sign follows from differentiating $\tilde u-\beta$. The same recursion is obtained by accumulating the upper-right block of $A_{\mathrm{LI}}$: the direct-product EqF transition exposes the physical bias-to-navigation sensitivity directly. Apply it as

$$
\Upsilon_k(\beta)\simeq
\hat\Upsilon_k
\operatorname{Exp}
\left(J_k(\beta-\hat\beta)\right).
$$

The six columns of $J_k$ are ordered $(b_a,b_\omega)$, exactly like `ConstantBias::vector()`. There is no intermediate $10\times10$ virtual-bias Jacobian and no final column permutation. For the standard Galilean PIM, bias evolution remains the responsibility of a separate factor.

## 8. From the Galilean uncertainty to a GTSAM factor

The factor consumes a nine-dimensional `NavState` residual rather than the ten-dimensional Gal3 local error. In their respective tangent orderings,

$$
e=(\delta\theta,\delta v,\delta p,\delta t)\in\mathbb R^{10},\qquad
n=(\delta\theta,\delta p,\delta v)\in\mathbb R^9.
$$

Define the required selection and permutation when constructing the factor:

$$
n=P_Ne,\qquad
P_N=
\begin{bmatrix}
I_3&0&0&0\\
0&0&I_3&0\\
0&I_3&0&0
\end{bmatrix}
\in\mathbb R^{9\times10}.
$$

The final column removes elapsed time, which is known exactly from the measurement timestamps. This projection applies to local errors, covariances, and Jacobians. The mean Galilean element is converted separately to `NavState` as $(R,p,v)$.

Let $\Sigma_e\in\mathbb R^{10\times10}$ be the conditional Galilean covariance propagated above. The factor's `NavState` residual covariance is

$$
\Sigma_N=P_N\Sigma_eP_N^\top\in\mathbb R^{9\times9}.
$$

Likewise, the first-order bias sensitivity of the local navigation error is

$$
J_N=P_NJ_k\in\mathbb R^{9\times6}.
$$

For the mean, first apply the Galilean bias correction and then construct

$$
\Delta X_{ij}=
\operatorname{NavState}
(\Delta R_{ij},\Delta p_{ij},\Delta v_{ij}).
$$

The known gravity and elapsed-time terms combine this delta with $X_i$ to form `predictedState_j`. The residual used by the existing GTSAM preintegration machinery is

$$
r_{ij}=\operatorname{Local}_{\mathrm{NavState}}
(X_j,\operatorname{predictedState}_j)
=X_j.\texttt{localCoordinates}(\operatorname{predictedState}_j).
$$

In the C++ interface this is `NavState::localCoordinates`; the argument order above is intentional because GTSAM's IMU residual asks how the measured state retracts to the prediction.

Conceptually this is a three-variable relation $(X_i,X_j,\beta_i)$. The `GalileanImuFactor` alias uses `ImuFactorT`, so each `NavState` is exposed as separate pose and velocity keys and the concrete graph factor has five keys. In both views there is one bias variable for the entire interval. Bias evolution remains a separate factor, exactly as for the standard `ImuFactor`.

When bias evolution and state--bias correlation should be modeled inside the IMU factor, use `PreintegratedCombinedMeasurementsG` with `GalileanCombinedImuFactor`. The Combined PIM retains the same Galilean mean while propagating a public $15\times15$ covariance over $(R,p,v,b_a,b_\omega)$. The Gal(3) clock coordinate is deterministic and is projected out exactly. The resulting six-way factor adds the six bias-random-walk residuals to the nine navigation residuals.

## 9. Exact prediction in a rotating navigation frame

Let $\omega_n$ be the constant angular velocity of the navigation frame, expressed in navigation-frame coordinates, and let $\Omega=[\omega_n]_\times$. In the $(R,v,p)$ display order, the continuous dynamics are

$$
\dot R=-\Omega R+R[\omega]_\times,\qquad
\dot v=g+Ra-2\Omega v-\Omega^2p,\qquad
\dot p=v.
$$

GTSAM stores the same state as `NavState(R,p,v)`. Define the mutually inverse transported-velocity maps in that order by

$$
\mathcal L_\Omega(R,p,v)=(R,p,\bar v),\quad \bar v=v+\Omega p,\qquad
\mathcal P_\Omega(R,p,\bar v)=(R,p,v),\quad v=\bar v-\Omega p.
$$

For $\theta_{ij}=-\omega_n\Delta t_{ij}$, form the exact $SO(3)$ kernels

$$
A_{ij}=\operatorname{Exp}(\theta_{ij}),\qquad
G^v_{ij}=J_L(\theta_{ij}),\qquad
G^p_{ij}=J_L(\theta_{ij})-\Gamma_2(\theta_{ij}),
$$

and the rotating-frame world increment, now written directly in `NavState` order,

$$
W^\Omega_{ij}=\operatorname{NavState}\left(
A_{ij},G^p_{ij}g\Delta t_{ij}^2,G^v_{ij}g\Delta t_{ij}
\right).
$$

The complete endpoint prediction preserves the same left-linear backbone:

$$
\boxed{X_j=\mathcal P_\Omega\!\left(
W^\Omega_{ij}\,\phi_{\Delta t_{ij}}(\mathcal L_\Omega(X_i))\,U_{ij}(\beta_i)
\right)}.
$$

Writing $U_{ij}=(\Delta R,\Delta p,\Delta v)$ and $W^\Omega_{ij}=(A,p_W,v_W)$ in GTSAM order makes the implementation explicit:

$$
R_j=AR_i\Delta R,\qquad
p_j=p_W+A(p_i+\bar v_i\Delta t+R_i\Delta p),
$$
$$
\bar v_j=v_W+A(\bar v_i+R_i\Delta v),\qquad
v_j=\bar v_j-\Omega p_j.
$$

This is an exact constant-rate rotating-frame transition: Coriolis and centrifugal terms arise from the lift, group composition, and projection rather than an appended acceleration correction. The body-frame increment $U_{ij}$, its covariance, and its bias correction are unchanged; only endpoint prediction and its Jacobians depend on $\omega_n$. In Python, enable this path with `params.setOmegaCoriolis(omega_n)`. Omitting it (or setting exactly zero) selects the inertial prediction.

## 10. Intended use

A user-facing workflow mirrors the existing IMU factors:

1. Construct preintegration parameters and a `PreintegratedImuMeasurementsG` at the current bias estimate. When the navigation frame rotates, call `setOmegaCoriolis` with its constant angular velocity in navigation-frame coordinates.
2. Integrate each accelerometer/gyroscope sample. The public API and all six-dimensional quantities use GTSAM order $(a,\omega)$; the lift places them in Gal3 order internally. Each call advances the Galilean mean on the right and propagates its conditional covariance.
3. Construct a `GalileanImuFactor` between the two pose/velocity pairs and the interval's bias key. During optimization, use the right-applied first-order bias correction rather than reintegrating immediately.
4. Add a separate factor between consecutive bias keys when bias random-walk evolution is part of the model. Alternatively, construct `PreintegratedCombinedMeasurementsG` and a `GalileanCombinedImuFactor` to model the second bias key, its random walk, and state--bias correlation inside one factor.

In schematic C++ form:

```cpp
PreintegratedImuMeasurementsG pim(params, biasHat);
for (const ImuSample& sample : samples) {
  pim.integrateMeasurement(sample.acceleration, sample.angularRate,
                           sample.deltaT);
}
graph.emplace_shared<GalileanImuFactor>(
    X(i), V(i), X(j), V(j), B(i), pim);
```

The Combined alternative has the corresponding form:

```cpp
PreintegratedCombinedMeasurementsG combinedPim(combinedParams, biasHat);
// Integrate the same samples as above.
graph.emplace_shared<GalileanCombinedImuFactor>(
    X(i), V(i), X(j), V(j), B(i), B(j), combinedPim);
```

The API is deliberately familiar; the essential differences are internal geometric choices. The executable Python example below follows the same sequence. It predicts a consistent endpoint only to demonstrate factor construction; in a real graph, $X_j$ is an optimizer variable rather than the prediction itself.

In [3]:
params = gtsam.PreintegrationParams.MakeSharedD(9.81)
params.setAccelerometerCovariance(1e-4 * np.eye(3))
params.setGyroscopeCovariance(1e-6 * np.eye(3))
params.setIntegrationCovariance(1e-8 * np.eye(3))

bias_hat = gtsam.imuBias.ConstantBias(
    np.array([0.01, -0.02, 0.03]),
    np.array([-0.01, 0.02, 0.01]),
)
pim = gtsam.PreintegratedImuMeasurementsG(params, bias_hat)

samples = [
    (np.array([0.2, -0.1, 9.7]), np.array([0.03, -0.02, 0.01]), 0.01),
    (np.array([0.1, 0.2, 9.8]), np.array([-0.01, 0.04, 0.02]), 0.02),
]
for measured_acc, measured_omega, delta_t in samples:
    pim.integrateMeasurement(measured_acc, measured_omega, delta_t)

print(f"Integrated {pim.deltaTij():.3f} s")
print("NavState residual covariance shape:", pim.preintMeasCov().shape)

Integrated 0.030 s
NavState residual covariance shape: (9, 9)


In [4]:
state_i = gtsam.NavState(
    gtsam.Rot3.RzRyRx(0.1, -0.2, 0.3),
    np.array([1.0, -2.0, 0.5]),
    np.array([0.4, -0.1, 0.2]),
)
state_j = pim.predict(state_i, bias_hat)

Xi, Vi = gtsam.symbol("x", 0), gtsam.symbol("v", 0)
Xj, Vj = gtsam.symbol("x", 1), gtsam.symbol("v", 1)
Bi = gtsam.symbol("b", 0)
factor = gtsam.GalileanImuFactor(Xi, Vi, Xj, Vj, Bi, pim)

graph = gtsam.NonlinearFactorGraph()
graph.add(factor)
error = factor.evaluateError(
    state_i.pose(), state_i.velocity(),
    state_j.pose(), state_j.velocity(), bias_hat,
)
print("Factor error at the predicted endpoint:", error)

Factor error at the predicted endpoint: [-5.69983774e-21 -4.35407115e-19  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00]


## References

- G. Delama, A. Fornasier, R. Mahony, and S. Weiss, [*Equivariant IMU Preintegration with Biases: a Galilean Group Approach*](https://arxiv.org/abs/2411.05548), IEEE Robotics and Automation Letters, 2025.
- [`EquivariantFilter`](EKF-variants.md#equivariantfilter): GTSAM's state, symmetry, orbit, input action, lift, and equivariant-error template vocabulary.
- [`gtsam::Gal3`](../../geometry/doc/Gal3.ipynb): group operations, exponential and logarithmic maps, and adjoints.
- [`Gal3ImuEKF`](Gal3ImuEKF.ipynb): Galilean state propagation in an invariant EKF.
- [`NavState`](NavState.ipynb): GTSAM's navigation manifold and local-coordinate convention.
- [`ImuFactor`](ImuFactor.ipynb): standard preintegration and factor-graph usage.